# CRISP-DM Titanic: executable walkthrough

## Goal
Reproduce every analytical lens used by the dashboard and inspect the evidence contract. The fixed seed and bundled 891-row Kaggle-compatible training file make this tutorial deterministic and offline-ready.

## Setup
Run from the project directory. The pipeline validates its input contract before analysis.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'crispdm').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from crispdm.pipeline import load_data, run_workflows

pd.set_option('display.max_colwidth', 80)

## Steps
### 1. Understand the data
Check grain, missingness, label prevalence, and grouped rates before choosing transformations.

In [2]:
passengers = load_data()
summary = {
    'shape': passengers.shape,
    'duplicate_ids': int(passengers.PassengerId.duplicated().sum()),
    'survival_rate': round(passengers.Survived.mean(), 3),
    'missing': passengers.isna().sum()[lambda s: s.gt(0)].to_dict(),
}
summary

{'shape': (891, 12),
 'duplicate_ids': 0,
 'survival_rate': 0.384,
 'missing': {'Age': 177, 'Cabin': 687, 'Embarked': 2}}

In [3]:
passengers.groupby(['Sex', 'Pclass']).Survived.agg(['count', 'mean']).round(3)

count   mean
Sex    Pclass              
female 1          94  0.968
       2          76  0.921
       3         144  0.500
male   1         122  0.369
       2         108  0.157
       3         347  0.135

### 2. Run preparation and all models
Numeric median imputation/scaling and categorical imputation/one-hot encoding happen in reusable pipelines. The target is excluded from unsupervised and retrieval inputs.

In [4]:
results = run_workflows()
results.keys()

Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  File "C:\Users\Liliya\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 217, in _count_physical_cores
    raise ValueError(
KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=4.
KMeans is known to have a memory leak on Windows with MKL, 

dict_keys(['metadata', 'understanding', 'clustering', 'anomalies', 'supervised', 'associations', 'lsh'])

### 3. Compare clustering and anomaly evidence
Silhouette selects among k=2..5. Isolation Forest's 5% contamination is a review policy, not a truth label.

In [5]:
pd.DataFrame(results['clustering']['profiles'])

,cluster,size,median_age,median_fare,survival_rate,female_share
0,0,150,14.5,31.27,0.520,0.600
1,1,741,30.0,12.35,0.356,0.302


In [6]:
pd.DataFrame(results['anomalies']['top']).head()

,passenger_id,name,age,fare,family_size,score
0,439,"Fortune, Mr. Mark",64.0,263.00,6,0.735
1,680,"Cardeza, Mr. Thomas Drake Martinez",36.0,512.33,2,0.705
2,259,"Ward, Miss. Anna",35.0,512.33,1,0.695
3,738,"Lesurer, Mr. Gustave J",35.0,512.33,1,0.695
4,864,"Sage, Miss. Dorothy Edith ""Dolly""",NaN,69.55,11,0.691


### 4. Evaluate supervised models
ROC-AUC measures ranking across thresholds, accuracy matches Kaggle's metric, and F1 balances precision and recall.

In [7]:
pd.DataFrame(results['supervised']['models']).T

,accuracy,f1,roc_auc
Logistic regression,0.776,0.699,0.842
Random forest,0.798,0.706,0.838


### 5. Inspect association rules
Support measures prevalence, confidence measures conditional frequency, and lift compares confidence with the consequent baseline.

In [8]:
pd.DataFrame(results['associations']['rules']).head(8)

,if,then,support,confidence,lift
0,"[embarked=C, fare=high]",class=1,0.095,0.914,3.770
1,"[fare=high, outcome=survived]",class=1,0.150,0.807,3.330
2,"[age=middle, fare=high]",class=1,0.089,0.760,3.133
3,"[class=1, embarked=C]",fare=high,0.095,1.000,3.000
4,"[class=1, outcome=survived]",fare=high,0.150,0.985,2.956
5,"[class=1, sex=female]",fare=high,0.103,0.979,2.936
6,"[fare=high, sex=female]",class=1,0.103,0.692,2.853
7,[fare=high],class=1,0.229,0.687,2.833


### 6. Inspect MinHash LSH
Banded signatures produce a candidate set; exact Jaccard similarity then ranks only those candidates. Candidate reduction trades exhaustive comparison for probabilistic recall.

In [9]:
pd.Series({k:v for k,v in results['lsh'].items() if k not in ['query','neighbors']}).to_frame('value')

,value
candidates_examined,60.000
total_rows,890.000
reduction,0.933


In [10]:
pd.DataFrame(results['lsh']['neighbors'])[['passenger_id','name','similarity']]

,passenger_id,name,similarity
0,744,"McNamee, Mr. Neal",1.0
1,705,"Hansen, Mr. Henrik Juul",1.0
2,665,"Lindqvist, Mr. Eino William",1.0
3,478,"Braund, Mr. Lewis Richard",1.0
4,443,"Petterson, Mr. Johan Emil",1.0


## Checks
The assertions below are compact reasonableness checks. The full suite additionally tests thresholds, APIs, and dashboard routes.

In [11]:
assert summary['shape'] == (891, 12)
assert summary['duplicate_ids'] == 0
assert all(0.5 < m['roc_auc'] <= 1 for m in results['supervised']['models'].values())
assert results['associations']['rules']
assert results['lsh']['candidates_examined'] < results['lsh']['total_rows']
print('All notebook checks passed.')

All notebook checks passed.


## Next steps
Use repeated cross-validation and calibration for a stronger performance estimate; compare LSH candidates with exhaustive top-k neighbors to measure recall; test log-fare sensitivity for anomalies; and add subgroup evaluation before any real operational use.

## Takeaways
The executed evidence shows two coarse clusters (silhouette 0.406), 45 policy-flagged anomalies, held-out ROC-AUC around 0.84, strong class/fare co-occurrence, and a 93.3% LSH candidate reduction. These methods answer different questions and none establishes causality.